In [ ]:
# Install required libraries
%pip install mlflow

# **1. Imports**

In [ ]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.style.use('ggplot')
import warnings
warnings.filterwarnings('ignore')

from scipy.io import loadmat
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight

# Models
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

# Metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, auc, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
    balanced_accuracy_score, cohen_kappa_score,
    log_loss, average_precision_score, precision_recall_curve
)

# MLFlow
import mlflow
from sklearn.preprocessing import label_binarize

# **2. Load Data**

In [2]:
# Load Data
elec_data = loadmat('/Users/seyedrumaiz/Library/CloudStorage/OneDrive-InformaticsInstituteofTechnology/DSGP/solar-panel-fault-mapping/datasets/dataset_elec.mat')
amb_data = loadmat('/Users/seyedrumaiz/Library/CloudStorage/OneDrive-InformaticsInstituteofTechnology/DSGP/solar-panel-fault-mapping/datasets/dataset_amb.mat')

vdc1 = elec_data['vdc1'].flatten()
vdc2 = elec_data['vdc2'].flatten()
idc1 = elec_data['idc1'].flatten()
idc2 = elec_data['idc2'].flatten()

irr = amb_data['irr'].flatten()
pvt = amb_data['pvt'].flatten()
f_nv = amb_data['f_nv'].flatten()

# Create DataFrame
df = pd.DataFrame({
    'vdc1': vdc1,
    'vdc2': vdc2,
    'idc1': idc1,
    'idc2': idc2,
    'irradiance': irr,
    'temperature': pvt,
    'fault_label': f_nv
})

# **3. Data Pre-processing and Splitting**

In [3]:
# Filter out unwanted labels
df = df[df['fault_label'] != 2].copy()

# Feature Engineering
df['power_string1'] = df['vdc1'] * df['idc1']
df['power_string2'] = df['vdc2'] * df['idc2']
df['total_power'] = df['power_string1'] + df['power_string2']
df['voltage_ratio'] = df['vdc1'] / df['vdc2']
df['current_ratio'] = df['idc1'] / df['idc2']

# Map fault names
fault_names = {0: 'Normal Operation', 1: 'Short-Circuit', 3: 'Open Circuit', 4: 'Shadowing'}
df['fault_label'] = df['fault_label'].map(fault_names)

In [4]:
# Prepare Features and Labels
X = df.drop(['fault_label'], axis=1)
y = df['fault_label']

In [5]:
# One-hot encode labels for multiclass metrics
ohe = OneHotEncoder()
y_encoded = ohe.fit_transform(y.values.reshape(-1, 1)).toarray()

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [6]:
# Compute class weights to handle imbalance
classes = np.unique(y_train)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weights = dict(zip(classes, weights))

print("Class weights:", class_weights)

Class weights: {'Normal Operation': np.float64(0.2931015301866836), 'Open Circuit': np.float64(56.585443037974684), 'Shadowing': np.float64(1.808509474131013), 'Short-Circuit': np.float64(56.82126484684309)}


In [7]:
# Scale features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# **4. Train and Evaluate Model**

In [ ]:
def train_and_evaluate_model(
    model,
    model_name,
    X_train,
    X_test,
    y_train,
    y_test,
    experiment_name="PV_Fault_Detection_Base_Models"
):
    """
    Trains a sklearn model, evaluates it using robust multiclass metrics,
    visualizes ROC & PR curves, logs everything to MLflow.

    Args:
        model: The model being trained and evaluated
        model_name: A name given for MLflow
        X_train: Training data
        X_test: Testing data
        y_train: y train data
        y_test: y test datas
        experiment_name: Name of the experiment being conducted.
    """

    mlflow.set_experiment(experiment_name)

    with mlflow.start_run(run_name=model_name):

        # Train model
        model.fit(X_train, y_train)

        # Predict
        y_pred = model.predict(X_test)

        if not hasattr(model, "predict_proba"):
            raise ValueError(f"{model_name} does not support predict_proba()")

        y_proba = model.predict_proba(X_test)

        # Metrics
        acc = accuracy_score(y_test, y_pred)
        bal_acc = balanced_accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average='macro')
        rec = recall_score(y_test, y_pred, average='macro')
        f1 = f1_score(y_test, y_pred, average='macro')
        kappa = cohen_kappa_score(y_test, y_pred)
        ll = log_loss(y_test, y_proba)

        # ROC and PR AUC
        y_test_bin = label_binarize(y_test, classes=model.classes_)

        # Get ROC AUC score
        roc_auc_macro = roc_auc_score(
            y_test_bin,
            y_proba,
            multi_class='ovr',
            average='macro'
        )

        # Get PR AUC
        pr_auc_macro = average_precision_score(
            y_test_bin,
            y_proba,
            average='macro'
        )

        # Document metrics
        metrics = {
            "accuracy": acc,
            "balanced_accuracy": bal_acc,
            "precision_macro": prec,
            "recall_macro": rec,
            "f1_macro": f1,
            "cohen_kappa": kappa,
            "log_loss": ll,
            "roc_auc_macro": roc_auc_macro,
            "pr_auc_macro": pr_auc_macro
        }

        for k, v in metrics.items():
            mlflow.log_metric(k, v)

        # ROC AUC curves
        plt.figure(figsize=(8, 6))

        for i, class_name in enumerate(model.classes_):
            fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_proba[:, i])
            auc_score = auc(fpr, tpr)

            plt.plot(fpr, tpr, lw=2,
                     label=f"{class_name} (AUC={auc_score:.3f})")

        plt.plot([0, 1], [0, 1], 'k--')
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title(f"ROC–AUC Curve ({model_name})")
        plt.legend()
        plt.grid(True)
        plt.tight_layout()

        mlflow.log_figure(plt.gcf(), "roc_auc_curve.png")
        plt.close()

        # PR Curves
        plt.figure(figsize=(8, 6))

        for i, class_name in enumerate(model.classes_):
            precision, recall, _ = precision_recall_curve(
                y_test_bin[:, i],
                y_proba[:, i]
            )

            plt.plot(
                recall,
                precision,
                lw=2,
                label=f"{class_name}"
            )

        plt.xlabel("Recall")
        plt.ylabel("Precision")
        plt.title(f"Precision–Recall Curve ({model_name})")
        plt.legend()
        plt.grid(True)
        plt.tight_layout()

        mlflow.log_figure(plt.gcf(), "precision_recall_curve.png")
        plt.close()

        # Confusion matrix
        cm = confusion_matrix(y_test, y_pred, labels=model.classes_)
        disp = ConfusionMatrixDisplay(cm, display_labels=model.classes_)
        disp.plot(cmap='Blues')
        plt.title(f"Confusion Matrix ({model_name})")
        plt.tight_layout()

        mlflow.log_figure(plt.gcf(), "confusion_matrix.png")
        plt.close()

        # Metrics table
        metrics_df = pd.DataFrame.from_dict(
            metrics, orient='index', columns=['Value']
        )

        metrics_path = "metrics_summary.csv"
        metrics_df.to_csv(metrics_path)
        mlflow.log_artifact(metrics_path)

        # Log model and display output
        mlflow.sklearn.log_model(model, name="model")

        # Display final summary
        print(f"\n ~ {model_name}")
        print(metrics_df)
        print("\nClassification Report:\n")
        print(classification_report(y_test, y_pred))

## **4.1. Random Forest**

In [ ]:
# Build random forest
rf_model = RandomForestClassifier(
    random_state=42,
    class_weight=class_weights,
    n_jobs=-1
)

# Train random forest
train_and_evaluate_model(
    rf_model,
    model_name="RandomForest_Base",
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test
)

2026/02/10 22:21:27 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas
2026/02/10 22:21:27 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables
2026/02/10 22:21:27 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types
2026/02/10 22:21:27 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.constraints
2026/02/10 22:21:27 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.defaults
2026/02/10 22:21:27 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.comments
2026/02/10 22:21:27 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/02/10 22:21:27 INFO alembic.runtime.migration: Will assume non-transactional DDL.



 ~ RandomForest_Base
                      Value
accuracy           0.999182
balanced_accuracy  0.998029
precision_macro    0.999007
recall_macro       0.998029
f1_macro           0.998518
cohen_kappa        0.996771
log_loss           0.002477
roc_auc_macro      0.999996
pr_auc_macro       0.999979

Classification Report:

                  precision    recall  f1-score   support

Normal Operation       1.00      1.00      1.00    232587
    Open Circuit       1.00      1.00      1.00      1205
       Shadowing       1.00      1.00      1.00     37694
   Short-Circuit       1.00      1.00      1.00      1200

        accuracy                           1.00    272686
       macro avg       1.00      1.00      1.00    272686
    weighted avg       1.00      1.00      1.00    272686



**Interpretation**

The Random Forest model demonstrates excellent performance across all evaluation metrics. With an overall AUC of near 100%, recal of 99.8%, precision of 99.9%, and accuracy of 99.9%, the model reliably predicts all fault types, even the less frequent ones. The per-class metrics show perfect/near-perfect precision, recall, F1-scores, indicating that the model correctly identifies all classes without misclassifying.

High values of Cohen's Kappa (99.7%), log loss (0.002), and PR-AUC (almost 100%) further confirm the model's robustness and its ability to distinguish between normal and different electrical faults accurately.

## **4.2. Naive Bayes**

In [ ]:
nb = GaussianNB()

train_and_evaluate_model(
    model=nb,
    model_name="NaiveBayes_Base",
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test
)

2026/02/10 00:05:07 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas
2026/02/10 00:05:07 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables
2026/02/10 00:05:07 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types
2026/02/10 00:05:07 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.constraints
2026/02/10 00:05:07 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.defaults
2026/02/10 00:05:07 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.comments
2026/02/10 00:05:08 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/02/10 00:05:08 INFO alembic.runtime.migration: Will assume non-transactional DDL.



 ~ NaiveBayes_Base
                      Value
accuracy           0.806147
balanced_accuracy  0.915189
precision_macro    0.838967
recall_macro       0.915189
f1_macro           0.853217
cohen_kappa        0.472466
log_loss           0.851378
roc_auc_macro      0.962895
pr_auc_macro       0.902834

Classification Report:

                  precision    recall  f1-score   support

Normal Operation       0.98      0.79      0.87    232587
    Open Circuit       0.98      1.00      0.99      1205
       Shadowing       0.41      0.88      0.56     37694
   Short-Circuit       0.99      0.99      0.99      1200

        accuracy                           0.81    272686
       macro avg       0.84      0.92      0.85    272686
    weighted avg       0.90      0.81      0.83    272686



**Interpretation**

The Naive Bayes model achieved an accuracy of 80.6% and a strong balanced accuracy of 91.5%, showing good performance across all classes despite imbalance. The high macro recall (0.92) and ROC-AUC (0.96) indicate effective fault detection and good class separability.

It performs very well on Open Circuit and Short-Circuit faults, but precision for the Shadowing class is low (0.41), meaning more false positives. Some Normal Operation samples are also misclassified, suggesting room for improvement in distinguishing between normal and shadowing conditions.

## **4.3. KNN**

In [ ]:
# Prepare the KNN
knn = KNeighborsClassifier(n_jobs=-1, weights='uniform')

# Train the KNN
train_and_evaluate_model(
    model=knn,
    model_name="KNN_Base",
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test
)


 ~ KNN_Base
                      Value
accuracy           0.998771
balanced_accuracy  0.996858
precision_macro    0.997667
recall_macro       0.996858
f1_macro           0.997263
cohen_kappa        0.995150
log_loss           0.011963
roc_auc_macro      0.999593
pr_auc_macro       0.998626

Classification Report:

                  precision    recall  f1-score   support

Normal Operation       1.00      1.00      1.00    232587
    Open Circuit       1.00      1.00      1.00      1205
       Shadowing       1.00      1.00      1.00     37694
   Short-Circuit       1.00      0.99      0.99      1200

        accuracy                           1.00    272686
       macro avg       1.00      1.00      1.00    272686
    weighted avg       1.00      1.00      1.00    272686



**Interpretation**

The KNN Base model achieved an accuracy of 99.88% and a balanced accuracy of 99.69%, indicating near-perfect performance across all classes. The macro precision, recall, F1-score, are all approximately 99.7%, with a ROC-AUC of 99.96%, showing excellent class seperability.

The classification report confirms almost perfect predictions for all fault types and Normal Operation, with only a slight drop in recall (99%) for Short-Circuit faults. Overall, the KNN model demonstrates outstanding performance with minimal misclassification.

## **4.4. AdaBoost**

In [ ]:
# Base estimator for AdaBoost
base_estimator = DecisionTreeClassifier(random_state=42, class_weight=class_weights)

# Initialize adaboost
adaboost = AdaBoostClassifier(
    estimator=base_estimator,
    random_state=42,
)

# Train and evaluate
train_and_evaluate_model(
    model=adaboost,
    model_name="AdaBoost_Base",
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test
)


 ~ AdaBoost_Base
                      Value
accuracy           0.998948
balanced_accuracy  0.996066
precision_macro    0.997794
recall_macro       0.996066
f1_macro           0.996928
cohen_kappa        0.995843
log_loss           1.073263
roc_auc_macro      0.997565
pr_auc_macro       0.994254

Classification Report:

                  precision    recall  f1-score   support

Normal Operation       1.00      1.00      1.00    232587
    Open Circuit       1.00      1.00      1.00      1205
       Shadowing       1.00      1.00      1.00     37694
   Short-Circuit       1.00      0.99      0.99      1200

        accuracy                           1.00    272686
       macro avg       1.00      1.00      1.00    272686
    weighted avg       1.00      1.00      1.00    272686



**Interpretation**

The AdaBoost base model achieved an accuracy of 99.89% and a balanced accuracy of 99.61%, indicating very strong and consistent performance across all classes. The macro precision, recall, and F1-score are all around 99.7%, while the ROC-AUC (99.76%) and PR-AUC (99.43%) confirm excellent class discrimination.

The classification report shows near-perfect predictions for all classes, with only a slight decrease in recall (99%) for Short-Circuit faults. Overall, AdaBoost demonstrates highly reliable fault classification with minimal misclassification.

## **4.5. Decision Tree**

In [ ]:
# Build base decision tree
dt_base = DecisionTreeClassifier(
    class_weight=class_weights,
    random_state=42
)

# Train and evaluate decision tree
train_and_evaluate_model(
    model=dt_base,
    model_name="DecisionTree_Base",
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test
)


 ~ DecisionTree_Base
                      Value
accuracy           0.998881
balanced_accuracy  0.996634
precision_macro    0.996687
recall_macro       0.996634
f1_macro           0.996659
cohen_kappa        0.995583
log_loss           0.040315
roc_auc_macro      0.997727
pr_auc_macro       0.993614

Classification Report:

                  precision    recall  f1-score   support

Normal Operation       1.00      1.00      1.00    232587
    Open Circuit       1.00      1.00      1.00      1205
       Shadowing       1.00      1.00      1.00     37694
   Short-Circuit       1.00      0.99      0.99      1200

        accuracy                           1.00    272686
       macro avg       1.00      1.00      1.00    272686
    weighted avg       1.00      1.00      1.00    272686



**Interpretation**


The Decision Tree base model achieved an accuracy of 99.89% and a balanced accuracy of 99.66%, showing excellent performance across all classes. The macro precision, recall, and F1-score are all approximately 99.7%, while the ROC-AUC (99.77%) and PR-AUC (99.36%) indicate strong class separability.

The classification report confirms near-perfect predictions for all classes, with only a slight drop in recall (99%) for Short-Circuit faults. Overall, the Decision Tree model demonstrates highly accurate and reliable fault classification with minimal errors.

## **4.6. CatBoost**

In [ ]:
# Prepare CatBoostClassifier
cat = CatBoostClassifier(
    loss_function="MultiClass",
    auto_class_weights="Balanced",  # CatBoost will balance automatically
    verbose=0,
    random_state=42
)

# Train and evaluate
train_and_evaluate_model(
    model=cat,
    model_name="CatBoost_Base",
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test
)


 ~ CatBoost_Base
                      Value
accuracy           0.997847
balanced_accuracy  0.998610
precision_macro    0.993829
recall_macro       0.998610
f1_macro           0.996205
cohen_kappa        0.991544
log_loss           0.005670
roc_auc_macro      0.999990
pr_auc_macro       0.999957

Classification Report:

                  precision    recall  f1-score   support

Normal Operation       1.00      1.00      1.00    232587
    Open Circuit       1.00      1.00      1.00      1205
       Shadowing       0.99      1.00      0.99     37694
   Short-Circuit       0.99      1.00      0.99      1200

        accuracy                           1.00    272686
       macro avg       0.99      1.00      1.00    272686
    weighted avg       1.00      1.00      1.00    272686



**Interpretation**

The CatBoost base model achieved an accuracy of 99.78% and an exceptionally high balanced accuracy of 99.86%, indicating outstanding and well-balanced performance across all classes. The macro recall (99.9%) and F1-score (99.6%) demonstrate strong fault detection capability, while the near-perfect ROC-AUC (0.99999) and PR-AUC (0.99996) confirm excellent class separability. The very low log loss (0.0057) indicates highly confident predictions.

The classification report shows almost perfect precision and recall for all classes, with only slight reductions in precision (0.99) for Shadowing and Short-Circuit faults. Overall, CatBoost delivers extremely accurate and reliable fault classification with minimal misclassification.

## **4.7. Logistic Regression**

In [ ]:
# Prepare logistic regression model
lr = LogisticRegression(
    class_weight=class_weights,
    random_state=42,
    multi_class="auto"
)

# Train and evaluate logistic regression
train_and_evaluate_model(
    model=lr,
    model_name="LogisticRegression_Base",
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test
)


 ~ LogisticRegression_Base
                      Value
accuracy           0.897464
balanced_accuracy  0.957419
precision_macro    0.885180
recall_macro       0.957419
f1_macro           0.909837
cohen_kappa        0.674570
log_loss           0.267257
roc_auc_macro      0.983432
pr_auc_macro       0.951625

Classification Report:

                  precision    recall  f1-score   support

Normal Operation       0.99      0.89      0.94    232587
    Open Circuit       1.00      1.00      1.00      1205
       Shadowing       0.58      0.94      0.72     37694
   Short-Circuit       0.97      1.00      0.99      1200

        accuracy                           0.90    272686
       macro avg       0.89      0.96      0.91    272686
    weighted avg       0.93      0.90      0.91    272686



**Interpretation**

The Logistic Regression base model achieved an accuracy of 89.75% and a strong balanced accuracy of 95.74%, indicating good overall performance across all classes despite imbalance. The macro recall (96%) and ROC-AUC (98%) show that the model effectively detects faults and separates classes well.

From the classification report, the model performs very well on Open Circuit and Short-Circuit faults with near-perfect recall. However, precision for the Shadowing class is relatively low (58)%, indicating more false positives, and Normal Operation recall (89%) shows some misclassification of normal samples. Overall, the model demonstrates solid fault detection but with moderate class overlap.

## **4.8. Extra Trees**

In [ ]:
# Prepare extra trees model
et = ExtraTreesClassifier(
    class_weight=class_weights,
    random_state=42
)

# Train and evaluate the model
train_and_evaluate_model(
    model=et,
    model_name="ExtraTrees_Base",
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test
)


 ~ ExtraTrees_Base
                      Value
accuracy           0.999157
balanced_accuracy  0.998431
precision_macro    0.998553
recall_macro       0.998431
f1_macro           0.998492
cohen_kappa        0.996670
log_loss           0.002507
roc_auc_macro      0.999995
pr_auc_macro       0.999972

Classification Report:

                  precision    recall  f1-score   support

Normal Operation       1.00      1.00      1.00    232587
    Open Circuit       1.00      1.00      1.00      1205
       Shadowing       1.00      1.00      1.00     37694
   Short-Circuit       1.00      1.00      1.00      1200

        accuracy                           1.00    272686
       macro avg       1.00      1.00      1.00    272686
    weighted avg       1.00      1.00      1.00    272686



**Interpretation**

The Extra Trees base model achieved an accuracy of 99.92% and a balanced accuracy of 99.84%, indicating exceptional and highly consistent performance across all classes. The macro precision, recall, and F1-score are all approximately 0.998, while the near-perfect ROC-AUC (0.999995) and PR-AUC (0.999972) demonstrate outstanding class separability. The extremely low log loss (0.0025) reflects very confident predictions.

The classification report confirms perfect or near-perfect precision and recall for all classes, with virtually no misclassification observed. Overall, the Extra Trees model delivers extremely accurate and robust fault classification performance.